# Mask R-CNN CTR mask prediction helper use case

This example shows how to use `pyhectr.mask_nn` for the two cases represented in the original notebooks:

- **DESY P07 / single crystal**: full detector TIFF/Varex images.
- **DESY P03 / polycrystalline**: Eiger HDF5/NumPy stacks with a detector crop because the full resolution was too large and CTR signals were small.

The predicted masks are saved as dense masks or bounding-box/coordinate `.npy` files and can then be loaded for reciprocal-space integration.

In [1]:
import os
import glob
import numpy as np
from detectron2.engine import DefaultPredictor

from pyhectr.xrd_geom import (
    generate_dilate,
    generate_halo,
    grow_or_shrink,
)

from pyhectr.read_plot import (
    compute_max_pixel_image,
    read_P07_imgs_with_metadata as read_imgs_with_metadata,
    read_metadata,
)

from pyhectr.io import (
    find_eiger_h5_files,
    load_eiger_h5_data,
    prepare_p03_eiger_stack,
    collect_npy_data_paths,
    load_and_sum_predictions,
)

from pyhectr.mask_nn import (
    # P03 / Eiger helpers
    prediction_output_dir,
    # preprocessing + inference
    build_mask_rcnn_cfg,
    run_prediction_for_npy_files,
    run_inference_with_default_predictor,
    process_images_array_parallel,
    which_method,
    which_method_intensity,
    possible_methods_intensity,
    # integration mask loading
)

# Polycrystalline case

## 1. P03 polycrystalline case: prepare cropped Eiger data

The notebook loads raw Eiger HDF5 data, removes very bright detector artefacts using a mean image threshold, and saves both full filtered and cropped stacks. The crop corresponds to the CTR signal use case.

In [ ]:
# P03 raw/processed paths
# Edit these for your beamtime/session.
data_path = "/asap3/petra3/gpfs/p03/2024/data/11018978/raw"
processed_data_path = "/asap3/petra3/gpfs/p03/2024/data/11018978/shared/scans_prepared"

NEED_PREPARING = False

if NEED_PREPARING:
    scan_folders = sorted(os.listdir(data_path))[2:]
    good_scans = scan_folders[23:31]

    for scan_name in good_scans:
        current_scan = os.path.join(data_path, scan_name, "eiger9m")
        _, h5_file, _ = find_eiger_h5_files(current_scan)
        h5_data = load_eiger_h5_data(h5_file)

        data_filtered, data_filtered_clip = prepare_p03_eiger_stack(
            h5_data,
            crop_y=slice(None, 2370),
            mean_threshold=4.2e9,
            clip_y=slice(1070, None),
            clip_x=slice(1030, None),
        )

        path2save = os.path.join(processed_data_path, scan_name)
        os.makedirs(path2save, exist_ok=True)
        np.save(os.path.join(path2save, f"{scan_name}.npy"), data_filtered)
        np.save(os.path.join(path2save, f"{scan_name}-clip.npy"), data_filtered_clip)

        print(f"saved {scan_name}: full={data_filtered.shape}, clip={data_filtered_clip.shape}")

## 2. Select prepared P03 `.npy` stacks


In [ ]:
data_paths = collect_npy_data_paths(processed_data_path)
data_paths_clip = [p for p in data_paths if "clip" in os.path.basename(p)]

# For a quick test, start with one scan.
data_paths_for_inference = data_paths_clip[:1]

data_paths_for_inference

## 3. Configure Mask R-CNN and select trained models

This reproduces the P03/P07 inference configuration with a single CTR mask class and Detectron2's Mask R-CNN R101/R50-FPN config. 

In [ ]:
output_dir = "../MaskRCNN"
model_dir = "../Mask_R_CNN_OUTPUT"

cfg = build_mask_rcnn_cfg(
    yaml_name="COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml", # or R101
    dataset_name="inference_dataset",
    score_thresh_test=0.05,
    device="cuda:0",
)

# Example of models selection rule
list_dir = sorted(os.listdir(model_dir))
models_filter = []
for name in list_dir:
    if "ANCHOR" not in name:
        if "frez2" in name and "50x3" in name:
            models_filter.append(name)

model_names = []
for m in models_filter:
    model_names.append(os.path.join(m, "model_final.pth"))
    model_names.append(os.path.join(m, "model_0039999.pth"))

len(model_names), model_names[:3]

### Output path construction



In [ ]:
# Preview where the first scan/model pair will be written.
example_output_dir = prediction_output_dir(
    output_dir,
    os.path.basename(data_paths_for_inference[0]).replace(".npy", ""),
    model_names[0],
    output_prefix="P03_clip_resized_2025",
)
example_output_dir


## 4. Run prediction on P03 cropped stacks


In [ ]:
outputs = run_prediction_for_npy_files(
    data_paths=data_paths_for_inference,
    model_names=model_names,
    model_dir=model_dir,
    output_dir=output_dir,
    cfg=cfg,
    full_size_predict=False,
    save_predicted_mask=False,
    save_prediction_bb=True,
    target_height=1024,
    target_width=1440,
    output_prefix="P03_clip_resized",
    n_jobs=-1,
    fps=1,
)

outputs[:1]

## 5. Load saved predictions for integration

In [ ]:
# Example: collect bb outputs from one inference directory.
inference_dir = "../"

current_models = [models_filter[0]]
model_dir_paths = [os.path.join(inference_dir, d) for d in current_models]

bb_files = []
for dir_path in model_dir_paths:
    bb_files.extend(
        os.path.join(dir_path, f)
        for f in os.listdir(dir_path)
        if f.endswith("bb.npy")
    )

binary_masks = load_and_sum_predictions(
    npy_files=bb_files,
    target_shape=(1024, 1440),
    score_thresh=0.01,
)

if binary_masks is not None:
    if binary_masks.max() > 1:
        binary_masks_sumed = (binary_masks / binary_masks.max()).astype(bool)
    else:
        binary_masks_sumed = binary_masks.astype(bool)

    print(binary_masks_sumed.shape, binary_masks_sumed.dtype)

## 6. Optional mask morphology before integration

Use this for growing/shrinking the predicted CTR mask or constructing an integration/background region.

In [ ]:
# Example for one frame or for a summed/max mask.
# frame_mask = binary_masks_sumed[0]
# grown = grow_or_shrink(frame_mask, expand=5, metric="pixels", struct="rect", return_ring=False)
# ring = grow_or_shrink(frame_mask, expand=5, metric="pixels", struct="rect", return_ring=True)

# Single crystal case

## 7. P07 single-crystal TIFF/Varex case

For P07 the TIFF images plus metadata sidecars and optionally uses the full detector ROI.

In [ ]:
InputDir = "/asap3/petra3/gpfs/p07/2024/data/11020129/raw/varex/"
Scan = "align_00841"

ROIy = slice(0, 2200)   # vertical
ROIx = slice(0, 2880)   # horizontal
img_path_pattern = InputDir + Scan + "/*[0-9].tif"

GREEDY = False
if GREEDY:
    # Greedy max-pixel image only:
    max_pixel_image = compute_max_pixel_image(img_path_pattern, roix=ROIx, roiy=ROIy)
else:
    # Full stack + metadata:
    images_array, omega_metadata = read_imgs_with_metadata(img_path_pattern, roix=ROIx, roiy=ROIy)
    images_array.shape, omega_metadata[:3]

## 8. Prepare P07 images and run the inference helper


In [ ]:
# Example preprocessing for P07 after loading images_array:
model_name = model_names[0]
channel_method = which_method(model_name, ["duplicate", "duplicate_var_prep", "custom_mix", "neighbor_slices"])
if "custom_mix" in channel_method:
    intensity_method = None
else:
    intensity_method = which_method_intensity(model_name, possible_methods_intensity)

prepared_images = process_images_array_parallel(
    images_array,
    ch_method=channel_method,
    method=intensity_method,
    n_jobs=-1,
)

cfg.MODEL.WEIGHTS = os.path.join(model_dir, model_name)
predictor = DefaultPredictor(cfg)
aggregated_mask = run_inference_with_default_predictor(
    images_array=prepared_images,
    predictor=predictor,
    cfg=cfg,
    output_movie_path="P07_prediction.mp4",
    target_size=(1024, 1440),
    save_mask=False,
    save_bb=True,
    output_bb_path="P07_prediction-bb.npy",
)